In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

"""
 1. Load data from data/processed/train_raw.csv
 2. Define preprocess:  

      Remove irrelevant columns
      Fill missing values
      Encode categorical variables
      Normalize or standardize continuous features
 3. Run simple assertion tests on processed data:
      No missing values
      Feature count matches expectation
 4. Save the output to processed/train_feat.csv
 5. Commit

"""


In [ ]:
# ── 2. Preprocessing function ─────────────────────────────────────────
def preprocess(df, scale=True):
    """
    Complete preprocessing pipeline:
    - Remove irrelevant columns
    - Fill missing values
    - Categorical variable One-Hot encoding
    - Standardize continuous features (optional)
    """
    df = df.copy()

    # 2-1 Remove irrelevant columns
    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    df.drop(columns=drop_cols, inplace=True)
    print(f"\n[Step1] Shape after removing irrelevant columns: {df.shape}")

    # 2-2 Fill missing values
    age_imputer = SimpleImputer(strategy='median')
    df['Age'] = age_imputer.fit_transform(df[['Age']]).ravel()

    emb_imputer = SimpleImputer(strategy='most_frequent')
    df['Embarked'] = emb_imputer.fit_transform(df[['Embarked']]).ravel()

    print(f"[Step2] Total missing values after imputation: {df.isnull().sum().sum()}")

    # 2-3 One-Hot encoding
    df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)
    print(f"[Step3] Shape after encoding: {df.shape}")
    print(f"        Current columns: {df.columns.tolist()}")

    # Convert bool columns to int (True→1, False→0) 
    bool_cols = df.select_dtypes(include='bool').columns 
    df[bool_cols] = df[bool_cols].astype(int)

    # 2-4 Standardize continuous features (optional)
    if scale:
        scale_cols = ['Age', 'Fare', 'SibSp', 'Parch']
        scaler = StandardScaler()
        df[scale_cols] = scaler.fit_transform(df[scale_cols])
        print(f"[Step4] Standardized columns: {scale_cols}")

    return df


In [ ]:
# ── 1. Load data ──────────────────────────────────────────
df = pd.read_csv('../data/processed/train_raw.csv')
print(f"Original data shape: {df.shape}")
print(f"Original column names: {df.columns.tolist()}")


原始数据维度: (891, 12)
原始列名: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [ ]:
# ── 3. Execute preprocessing ─────────────────────────────────────────
df_feat = preprocess(df, scale=True)
print(f"\nProcessed data shape: {df_feat.shape}")
print(df_feat.head())



[Step1] 删除无关列后维度: (891, 8)
[Step2] 填补后缺失值总数: 0
[Step3] 编码后维度: (891, 9)
        当前列名: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']
[Step4] 已标准化列: ['Age', 'Fare', 'SibSp', 'Parch']

处理后数据维度: (891, 9)
   Survived  Pclass       Age     SibSp     Parch      Fare  Sex_male  \
0         0       3 -0.565736  0.432793 -0.473674 -0.502445         1   
1         1       1  0.663861  0.432793 -0.473674  0.786845         0   
2         1       3 -0.258337 -0.474545 -0.473674 -0.488854         0   
3         1       1  0.433312  0.432793 -0.473674  0.420730         0   
4         0       3  0.433312 -0.474545 -0.473674 -0.486337         1   

   Embarked_Q  Embarked_S  
0           0           1  
1           0           0  
2           0           1  
3           0           1  
4           0           1  


In [24]:
print(df_feat.shape, df_feat.columns.tolist())

(891, 9) ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']


In [ ]:
# ── 4. Assertion tests ───────────────────────────────────────────
print("\n── Assertion tests ──")

# Test 1: No missing values
assert df_feat.isnull().sum().sum() == 0, \
    "[FAIL] Missing values exist"
print("[PASS] No missing values")

# Test 2: Feature count meets expected
#  Original 12 columns
# - Remove 4 columns (PassengerId, Name, Ticket, Cabin) = 8 columns
# - Sex: 1 column → 1 column (drop_first removes Female)
# - Embarked: 1 column → 2 columns (drop_first removes C, keeps Q/S)
# Total: 4(numeric) + 1(Sex_male) + 2(Embarked_Q/S) + 1(Survived) + 1(Pclass) = 9 columns
EXPECTED_COLS = 9
EXPECTED_COLS = 9
assert df_feat.shape[1] == EXPECTED_COLS, \
    f"[FAIL] Feature count mismatch: expected {EXPECTED_COLS}, got {df_feat.shape[1]}"
print(f"[PASS] Feature count correct: {df_feat.shape[1]} columns")

# Test 3: Target column exists and has correct domain
assert set(df_feat['Survived'].unique()).issubset({0, 1}), \
    "[FAIL] Survived column domain is abnormal"
print("[PASS] Survived column domain normal: {0, 1}")


# Test 4: Row count unchanged
assert len(df_feat) == len(df), \
    f"[FAIL] Row count changed: original {len(df)} rows, now {len(df_feat)} rows"
print(f"[PASS] Row count unchanged: {len(df_feat)} rows")



── 断言测试 ──
[PASS] 无缺失值
[PASS] 特征数量正确：9 列
[PASS] Survived 列值域正常 {0, 1}
[PASS] 行数不变：891 行


In [ ]:
# ── 5. Save results ───────────────────────────────────────────
import os
os.makedirs('../data/processed', exist_ok=True)
df_feat.to_csv('../data/processed/train_feat.csv', index=False)
print("\n✅ Saved: data/processed/train_feat.csv")
print(f"   Final shape: {df_feat.shape}")



✅ 已保存：data/processed/train_feat.csv
   最终维度：(891, 9)
